In [1]:
from pymongo import MongoClient

# 1. 連接到遠端 MongoDB
mongo_client = MongoClient("mongodb+srv://localhost:12027/")  # 替換為你的遠端 MongoDB 連線字串

# 2. 列出所有資料庫名稱
database_names = mongo_client.list_database_names()

# 選擇資料庫
db = mongo_client["sample_traffic"]

# 選擇集合 (collection)
collection = db["myCollection"]

## **資料庫查詢**

你可以根據這個 MongoDB 數據庫，從以下 **10 大問題類別** 進行查詢與分析：  

### **1. 物件偵測統計**  
📌 **問題示例**：「某個 frame_number 中，偵測到哪些物件？」  
➡️ 這類問題關心某個影格偵測到了哪些 `class_name`。  

In [2]:
## Intent_1
collection.distinct("class_name", {"frame_number": 287})


['car', 'person', 'truck']

### **2. 物件數量分析**  
📌 **問題示例**：「哪種類型的物件 (class_name) 出現最多次？」  
➡️ 這類問題統計 `class_name` 出現的頻率，分析數據分佈。  


In [3]:
## Intent_2
list(collection.aggregate([
    {"$group": {"_id": "$class_name", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}
]))


[{'_id': 'person', 'count': 2737},
 {'_id': 'car', 'count': 2114},
 {'_id': 'motorcycle', 'count': 425},
 {'_id': 'truck', 'count': 332},
 {'_id': 'bus', 'count': 54}]

### **3. 影格 (frame) 變化趨勢**  
📌 **問題示例**：「某個 instance_id 在多少影格 (frame_number) 中出現？」  
➡️ 這類問題關心物件是否在多個影格中持續追蹤。  


In [4]:
## Intent_3
print("{}".format(collection.distinct("frame_number", {"instance_id": 7})))


[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 22


### **4. 信心度 (confidence) 分析**  
📌 **問題示例**：「影像中所有物件的平均信心度是多少？」  
➡️ 這類問題分析模型對偵測結果的置信度表現。  


In [5]:
## Intent_4
intent4 = list(collection.aggregate([
    {"$group": {"_id": None, "avg_confidence": {"$avg": "$confidence"}}}
]))

intent4_tmp = list(collection.aggregate([
    {"$group": {"_id": "$class_name", "avg_confidence": {"$avg": "$confidence"}}},
    {"$sort": {"avg_confidence": -1}}
]))

intent4.extend(intent4_tmp)

for item in list(intent4):
    print("{}".format(item))


{'_id': None, 'avg_confidence': 0.44620305467008786}
{'_id': 'car', 'avg_confidence': 0.5094623450813648}
{'_id': 'bus', 'avg_confidence': 0.4303685650229454}
{'_id': 'person', 'avg_confidence': 0.42704569617511584}
{'_id': 'motorcycle', 'avg_confidence': 0.3467353533471332}
{'_id': 'truck', 'avg_confidence': 0.33124036121978817}



### **5. 物件位置與範圍 (Bounding Box) 分析**  
📌 **問題示例**：「bounding box x1 在 200-300 之間的物件有多少？」  
➡️ 這類問題分析物件的空間分佈。  


In [6]:
## Intent_5

collection.count_documents({"box.x1": {"$gte": 200, "$lte": 250}})


508


### **6. 物件移動與追蹤**  
📌 **問題示例**：「某個 instance_id 的 bounding box 位置變化情況？」  
➡️ 這類問題追蹤單個物件在不同影格中的移動軌跡。  



In [7]:
import pandas as pd

intent6 = collection.find({"instance_id": 7}, {"frame_number": 1, "box": 1, "_id": 0})
# print((list(intent6)))

flattened_data = []

for entry in list(intent6):
    flattened_entry = {
        "frame_number": entry["frame_number"],
        "box_x1": entry['box']['x1'],
        "box_y1": entry['box']['y1'],
        "box_x2": entry['box']['x2'],
        "box_y2": entry['box']['y2'],
    }
#     print(flattened_entry)
    flattened_data.append(flattened_entry)

df = pd.DataFrame(flattened_data)
print(df.head(5))

   frame_number      box_x1     box_y1      box_x2      box_y2
0             1  295.096222  98.485504  305.272552  137.333374
1             2  295.190002  98.465538  305.413086  137.445236
2             3  295.254669  98.388397  305.462433  137.489685
3             4  295.272369  98.462189  305.326050  137.483139
4             5  295.263367  98.533112  305.255280  137.503021


### **7. 影像品質與異常偵測**  
📌 **問題示例**：「哪些影格的 confidence 過低，可能需要重新標註？」  
➡️ 這類問題可以用來找出可能的錯誤偵測或低品質結果。  



In [8]:
intent7 = collection.aggregate([
    {"$match": {"confidence": {"$lt": 0.3}}},
    {"$group": {"_id": "$frame_number", "objects": {"$push": "$$ROOT"}}},
    {"$sort": {"_id": 1}}  # 按 frame_number 升序排列
])

for item in list(intent7)[:10]:
    print("Frame Number: {}".format(item['_id']))
    for obj in item['objects']:
        print("    Class: {}, Instance ID: {}, Confidence: {:.3f}".format(
            obj['class_name'], obj['instance_id'], obj['confidence']
        ))

# **`$group`**: 根據 `frame_number` 分組，並將每個影格中的物件 (`$$ROOT` 表示整個文檔) 加入 `objects` 陣列中。

Frame Number: 1
    Class: bus, Instance ID: 23, Confidence: 0.298
    Class: person, Instance ID: 24, Confidence: 0.284
    Class: truck, Instance ID: 25, Confidence: 0.271
Frame Number: 2
    Class: car, Instance ID: 20, Confidence: 0.232
    Class: motorcycle, Instance ID: 22, Confidence: 0.294
    Class: person, Instance ID: 24, Confidence: 0.267
    Class: car, Instance ID: 25, Confidence: 0.111
Frame Number: 3
    Class: car, Instance ID: 9, Confidence: 0.296
    Class: car, Instance ID: 17, Confidence: 0.136
    Class: car, Instance ID: 20, Confidence: 0.163
    Class: person, Instance ID: 24, Confidence: 0.273
    Class: car, Instance ID: 25, Confidence: 0.176
Frame Number: 4
    Class: car, Instance ID: 9, Confidence: 0.140
    Class: motorcycle, Instance ID: 17, Confidence: 0.169
    Class: car, Instance ID: 20, Confidence: 0.139
    Class: person, Instance ID: 24, Confidence: 0.273
    Class: car, Instance ID: 25, Confidence: 0.164
Frame Number: 5
    Class: car, Instance ID

### **8. 特定類別物件篩選**  
📌 **問題示例**：「在哪些影格中偵測到 ‘bus’？」  
➡️ 這類問題查詢特定 `class_name` 的出現情況。  



In [9]:
print("*Bus* detected in Frame Number:\n{}".format(", ".join(str(x) for x in collection.distinct("frame_number", {"class_name": "bus"}))))

*Bus* detected in Frame Number:
1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 64, 65


## **9. 影格關聯性分析**
📌 **問題示例**：「哪些影格同時偵測到 "car" 和 "person"？」  
➡️ 這類問題查詢多種類物件的共同出現情況。

In [10]:
intent9 = list(collection.aggregate([
    {"$match": {"class_name": {"$in": ["car", "person"]}}},
    {"$group": {"_id": "$frame_number", "count": {"$sum": 1}}},
    {"$match": {"count": {"$gte": 2}}},
    {"$project": {"frame_number": "$_id", "_id": 0}}
]))

print("*Car* and *Person* detected in Frame Number:\n{}".format(", ".join(str(x['frame_number']) for x in intent9)))

*Car* and *Person* detected in Frame Number:
102, 62, 177, 254, 26, 272, 145, 284, 45, 51, 186, 268, 154, 183, 221, 39, 194, 205, 259, 159, 214, 90, 250, 36, 75, 187, 53, 296, 134, 184, 199, 195, 164, 92, 270, 291, 69, 105, 68, 207, 185, 213, 218, 46, 78, 256, 271, 263, 116, 99, 189, 48, 59, 255, 101, 125, 180, 251, 245, 96, 123, 240, 8, 155, 110, 153, 15, 253, 146, 80, 281, 124, 70, 30, 74, 276, 66, 9, 147, 176, 158, 122, 149, 100, 44, 97, 115, 273, 249, 224, 188, 201, 22, 242, 128, 112, 12, 196, 118, 208, 239, 287, 223, 219, 52, 156, 106, 152, 23, 209, 258, 56, 61, 292, 27, 19, 151, 173, 261, 129, 10, 150, 111, 220, 4, 21, 18, 16, 216, 167, 132, 72, 170, 143, 179, 222, 244, 206, 94, 247, 81, 265, 289, 7, 38, 283, 119, 175, 88, 274, 104, 144, 230, 168, 157, 243, 246, 264, 278, 86, 108, 217, 288, 190, 140, 294, 293, 169, 262, 211, 298, 225, 285, 215, 3, 24, 65, 160, 269, 103, 257, 49, 40, 232, 135, 172, 127, 50, 37, 58, 77, 91, 139, 95, 109, 197, 54, 55, 64, 137, 43, 231, 33, 98, 76, 1

## **10. 特定條件篩選**
📌 **問題示例**：「frame_number > 280，且 confidence > 0.7 的物件有哪些？」  
➡️ 查詢方式：篩選 `frame_number > 280` 且 `confidence > 0.7`。

In [11]:
intent10 = collection.aggregate([
    {"$match": {"frame_number": {"$gt": 280},  "confidence": {"$gt": 0.7}}},
    {"$group": {"_id": "$frame_number", "objects": {"$push": "$$ROOT"}}},
    {"$sort": {"_id": 1}}  # 按 frame_number 升序排列
])
for item in intent10:
    print("Frame Number: {}".format(item['_id']))
    for obj in item['objects']:
        print("    Class: {}, Instance ID: {}, Confidence: {:.4f}".format(
            obj['class_name'], obj['instance_id'], obj['confidence']
        ))

Frame Number: 281
    Class: car, Instance ID: 3, Confidence: 0.8718
    Class: car, Instance ID: 571, Confidence: 0.9164
Frame Number: 282
    Class: car, Instance ID: 3, Confidence: 0.8803
    Class: car, Instance ID: 571, Confidence: 0.9137
Frame Number: 283
    Class: car, Instance ID: 3, Confidence: 0.8828
    Class: car, Instance ID: 571, Confidence: 0.9115
Frame Number: 284
    Class: car, Instance ID: 3, Confidence: 0.8739
    Class: car, Instance ID: 571, Confidence: 0.8901
Frame Number: 285
    Class: car, Instance ID: 3, Confidence: 0.8431
    Class: car, Instance ID: 571, Confidence: 0.9018
    Class: person, Instance ID: 616, Confidence: 0.7579
Frame Number: 286
    Class: car, Instance ID: 3, Confidence: 0.8494
    Class: car, Instance ID: 571, Confidence: 0.9064
Frame Number: 287
    Class: car, Instance ID: 3, Confidence: 0.8266
    Class: car, Instance ID: 571, Confidence: 0.8833
    Class: person, Instance ID: 616, Confidence: 0.7479
Frame Number: 288
    Class: car, 